## **Baseline Model Evaluation**

This notebook evaluates the winning solution from the [AICrowd Airborne Object Tracking Challenge](https://www.aicrowd.com/challenges/airborne-object-tracking-challenge) (2021) as the project baseline. The solution, developed by Dmytro Poplavskiy, uses a segmentation-based detection and tracking pipeline (SegTracker) with an ensemble of four deep learning models: HRNet32, HRNet48, GerNet-M with BiFPN, and DLA60.

Inference is performed across the entire custom hard dataset subset [outlined below](#dataset), with results presented both as an overall set of benchmark statistics and as performance metrics for each data subset.

**Evaluation Metrics:**
- **AFDR:** Airborne Flight Detection Rate. The primary detection metric.
- **FPPI:** False Positives Per Image. Must be ≤ 0.0005 to be ranked.
- **HFAR:** Horizontal False Alarm Rate. Must be ≤ 0.5 to be ranked.
- **EDR:** Encounter Detection Rate. Tracking-based metric across all horizon scenarios.

> ⚠️ ***Note:*** *Inference requires a CUDA-capable GPU. The `map_location` fix in `seg_tracker.py` allows CPU loading of weights for inspection, but full inference at acceptable speed requires GPU. Estimated runtime on GPU: ~2–3 FPS per flight sequence.*

In [1]:
# Import Required Libraries.
import os
import sys

import json
import subprocess

import pandas as pd
from pathlib import Path

from glob import glob
from logging_module import LogLevel, log

# Set Working Directory to Project Root.
os.chdir("..") 

# Add Project Root to Path.
sys.path.insert(0, os.getcwd())

# Add Core Metric Folders to System Path.
sys.path.insert(0, os.path.join(os.getcwd(), "core/metrics"))


---

<br/>

## **Dataset**

The custom dataset used in this notebook is a curated hard subset of the full Airborne Object Tracking (AOT) dataset, specifically designed to stress-test model performance under the most demanding real-world conditions. Flights were selected to isolate encounters that represent the greatest challenge for vision-based detection and tracking systems.

**Isolation Criteria - encounters were selected for:**
- **Small Targets:** Airborne objects occupying only a few pixels per frame.
- **Multiple Targets:** Scenes containing more than one airborne object simultaneously.
- **Poor Visibility:** Flights affected by sun glare, adverse lighting, or challenging camera angles.

Together these conditions reflect the edge cases where sense-and-avoid systems are most likely to fail, and where reliable early detection is most critical for safe autonomous flight.

### **Airborne Object Tracking (AOT) - Hard Subset Summary**

| Metric                   | Value   | Description                                                             |
|--------------------------|--------:|-------------------------------------------------------------------------|
| **Total Flights**        | **1618**| **Total number of flight sequences in the dataset**                     |
| Small Object Flights     | 662     | Flights containing at least one object with bounding box area ≤ 36.0px².|
| Below Horizon Flights    | 1053    | Flights containing at least one object at or below the horizon.         |
| Multi-Object Flights     | 627     | Flights containing more than one airborne object in at least one frame. |
| No Object Frames         | 354     | Flights containing at least one frame with no airborne object present.  |
| *Avg. Sequence Length*   | *17.77s*| *Average duration of each flight sequence.*                             |

In [ ]:
# Import Data Downloader Libraries.
from src.data.download_data import main as downloadData

# Set Path to Desired Evaluation Dataset.
DATASET_PATH = "configs/datasets/hard_dataset_flights.txt"

# Download the Selected Flight Sequences.
downloadData(DATASET_PATH)

In [2]:
# Set Dataset Ground Truth Folders.
DATASET_PARTS = [
    "data/part1/ImageSets",
    "data/part2/ImageSets",
    "data/part3/ImageSets",
]

# Set Corresponding Inference Image Folders.
IMAGE_PARTS = [
    "data/part1/Images",
    "data/part2/Images",
    "data/part3/Images",
]

# Set Data Folder Names Used During Inference.
DATASET_ENVS = [
    "run_part1",
    "run_part2",
    "run_part3",
]

# Set Output Results Folders (Must Match DATASET_ENV Names Above).
RESULTS_DIRS = [
    "data/results/baseline/run_part1",
    "data/results/baseline/run_part2",
    "data/results/baseline/run_part3",
]

# Aggregated Summaries Output Folder.
SUMMARIES_DIR = "data/results/baseline/summaries"

# Path to the Metrics Runner Script.
METRICS_SCRIPT = "core/metrics/run_airborne_metrics.py"

# Minimum Score Threshold.
MIN_SCORE = 0.985

# Minimum Track Length Passed to Metrics Script.
# NOTE: Track length filtering is already applied during inference. Pass 0 here 
# to avoid double-filtering which causes NaN assertion errors in the metrics.
MIN_TRACK_LEN = 0

# Print Summary Statement.
log(LogLevel.INFO, "baseline:config", "Configuration Loaded.")
print(f"  Dataset Parts : {len(DATASET_PARTS)}")
print(f"  Results Dirs  : {RESULTS_DIRS}")
print(f"  Summaries Dir : {SUMMARIES_DIR}")
print(f"  Min Score     : {MIN_SCORE}")
print(f"  Min Track Len : {MIN_TRACK_LEN}")

<INFO ><baseline:config> Configuration Loaded.
  Dataset Parts : 3
  Results Dirs  : ['data/results/baseline/run_part1', 'data/results/baseline/run_part2', 'data/results/baseline/run_part3']
  Summaries Dir : data/results/baseline/summaries
  Min Score     : 0.985
  Min Track Len : 0



---

<br/>

## **Evaluate Baseline Model**

### **Run Baseline Inference**

Run `seg_test.py` across each dataset part sequentially and save the results to the configured output directories.

> ⚠️ **This cell requires a CUDA GPU and will take several hours to complete.** Skip this cell if inference has already been run and results are present in `RESULTS_DIRS`.

In [6]:
# Import Data Downloader Libraries.
from core.metrics.run_airborne_metrics import get_next_run_dir

# Run Inference On Each Dataset Part.
for i, (image_dir, results_dir) in enumerate(zip(IMAGE_PARTS, RESULTS_DIRS)):
    # Set Part Name.
    part_name = f"Part {i+1}"

    # Auto-Increment Run Folder so Previous Results are Never Overwritten.
    run_name = get_next_run_dir(results_dir)

    # Print Current Run Info.
    log(LogLevel.INFO, "baseline:inference", "Running Inference on %s...", part_name)
    print(f"  Images  : {image_dir}")
    print(f"  Results : {results_dir}")
    print(f"  Run Folder : {run_name}")
    print(f"{'-'*58}")

    # Ensure Results Directory Exists. 
    os.makedirs(results_dir, exist_ok=True)

    # Set the Variables to Correct Paths.
    env = os.environ.copy()
    env["TEST_DATASET_PATH"] = image_dir
    env["INFERENCE_OUTPUT_PATH"] = results_dir
    env["DATASET_ENV"] = run_name

    # Run the Basline Model Inference.
    result = subprocess.run(
        [sys.executable, "-m", "src.baseline.seg_test"],
        env=env,
        capture_output=False
    )

    # Print Message on Completion. 
    if result.returncode == 0:
        log(LogLevel.INFO, "baseline:inference", "%s Inference Complete.\n", part_name)
    else:
        log(LogLevel.ERROR, "baseline:inference", "%s Inference Failed with Return Code %s.\n", part_name, result.returncode)

<INFO ><baseline:inference> Running Inference on Part 1...
  Images  : data/part1/Images
  Results : data/results/baseline/run_part1
  Run Folder : run0
----------------------------------------------------------
hrnet_w32 upscale: nearest comb outputs: 512
Feature channels: [32, 64, 128, 256]
Using custom backbone gernet_m
{'name': 'tf_efficientdet_d2', 'backbone_name': 'gernet_m', 'backbone_args': {'in_chans': 2}, 'backbone_indices': None, 'image_size': [512, 512], 'num_classes': 2, 'min_level': 3, 'max_level': 7, 'num_levels': 5, 'num_scales': 3, 'aspect_ratios': [[1.0, 1.0], [1.4, 0.7], [0.7, 1.4]], 'anchor_scale': 4.0, 'pad_type': 'same', 'act_type': 'swish', 'norm_layer': None, 'norm_kwargs': {'eps': 0.001, 'momentum': 0.01}, 'box_class_repeats': 3, 'fpn_cell_repeats': 5, 'fpn_channels': 112, 'separable_conv': True, 'apply_resample_bn': True, 'conv_after_downsample': False, 'conv_bn_relu_pattern': False, 'use_native_resize_op': False, 'downsample_type': 'max', 'upsample_type': 'ne

  1%|          | 1/105 [10:29<18:11:46, 629.87s/it]
Traceback (most recent call last):
  File "/usr/lib/python3.8/runpy.py", line 194, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.8/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/home/busatmd021/projects/2026/Vision-Based-Sense-and-Avoid/src/baseline/seg_test.py", line 211, in <module>
    submission.run()
  File "/home/busatmd021/projects/2026/Vision-Based-Sense-and-Avoid/src/baseline/evaluator/airborne_detection.py", line 131, in run
    raise e
  File "/home/busatmd021/projects/2026/Vision-Based-Sense-and-Avoid/src/baseline/evaluator/airborne_detection.py", line 125, in run
    self.evaluation()
  File "/home/busatmd021/projects/2026/Vision-Based-Sense-and-Avoid/src/baseline/evaluator/airborne_detection.py", line 116, in evaluation
    self.inference(flight)
  File "/home/busatmd021/projects/2026/Vision-Based-Sense-and-Avoid/src/baseline/seg_test.py", line 120, 

Traceback (most recent call last):
  File "/home/busatmd021/projects/2026/Vision-Based-Sense-and-Avoid/src/baseline/evaluator/airborne_detection.py", line 125, in run
    self.evaluation()
  File "/home/busatmd021/projects/2026/Vision-Based-Sense-and-Avoid/src/baseline/evaluator/airborne_detection.py", line 116, in evaluation
    self.inference(flight)
  File "/home/busatmd021/projects/2026/Vision-Based-Sense-and-Avoid/src/baseline/seg_test.py", line 120, in inference
    results = self.tracker.predict(image=frame, prev_image=prev_frame)
  File "/home/busatmd021/projects/2026/Vision-Based-Sense-and-Avoid/src/baseline/seg_tracker/seg_tracker.py", line 279, in predict
    detected_objects, prev_tr = self.detector.detect_objects(image=image, prev_image=prev_image)
  File "/home/busatmd021/projects/2026/Vision-Based-Sense-and-Avoid/src/baseline/seg_tracker/seg_tracker.py", line 223, in detect_objects
    prev_tr = self.estimate_transformation_full(prev_img=prev_image, img=image)
  File "/h

  0%|          | 0/173 [00:00<?, ?it/s]

KeyboardInterrupt: 

### **Run Metrics Evaluation**

Run the official AICrowd evaluation metrics against the ground truth for each dataset part, saving per-part summaries to `SUMMARIES_DIR`.

In [ ]:
# Ensure the Summaries Directory Exists. 
os.makedirs(SUMMARIES_DIR, exist_ok=True)

# Run Metrics for Each Part.
for i, (dataset_dir, results_dir) in enumerate(zip(DATASET_PARTS, RESULTS_DIRS)):
    # Set Part Name.
    part_name = f"Part {i+1}"

    # Set Results Folder Name.
    results_file_dir = os.path.join(results_dir, run_name)
    
    # Print Current Run Info.
    log(LogLevel.INFO, "baseline:evaluation", "Evaluating %s...", part_name)
    print(f"  Ground Truth : {dataset_dir}")
    print(f"  Results      : {results_file_dir}")

    # Check Results Exist Before Evaluating.
    result_files = (
        glob(os.path.join(results_file_dir, "*.json")) +
        glob(os.path.join(results_file_dir, "*.csv"))
    )
    if not result_files:
        log(LogLevel.WARNING, "baseline:evaluation", "No Result Files Found in %s - Skipping.", results_file_dir)
        log(LogLevel.WARNING, "baseline:evaluation", "Run Inference Module First.")
        continue

    # Save Results for Each Part Separately.
    part_summaries_dir = os.path.join(SUMMARIES_DIR, part_name)
    os.makedirs(part_summaries_dir, exist_ok=True)

    # Run the Metrics Calculating Script. 
    result = subprocess.run(
        [
            sys.executable, METRICS_SCRIPT,
            "--dataset-folder", dataset_dir,
            "--results-folder", results_file_dir,
            "--summaries-folder", part_summaries_dir,
            "--min-score", str(MIN_SCORE),
            "--min-track-len", str(MIN_TRACK_LEN),
        ],
        capture_output=True,
        text=True
    )

    # Print Message on Completion. 
    if result.returncode == 0:
        log(LogLevel.INFO, "baseline:evaluation", "%s Metrics Complete.\n", part_name)
    else:
        log(LogLevel.ERROR, "baseline:inference", "%s Metrics Failed with Error:\nSTDOUT:\n%s\nSTDERR:\n%s\n", 
            part_name, result.stdout, result.stderr)

### **Aggregate Results**

Load the per-part summary JSON files and aggregate them into a single set of benchmark statistics across the full custom dataset.

In [ ]:
# Initialise an Empty List to Store Summaries from All Parts.
all_summaries = []

# Load All Summary JSON Files Across All Parts.
for i in range(1, 4):
    # Construct the Directory Path for the Current Part.
    part_summaries_dir = os.path.join(SUMMARIES_DIR, f"part{i}")

    # Get a List of All JSON Files in the Current Part's Directory.
    summary_files = glob(os.path.join(part_summaries_dir, "*summary*.json"))

    # Loop through Each Summary File Found.
    for f in summary_files:
        with open(f) as jf:
            summary = json.load(jf)
            summary["part"] = f"part{i}"
            all_summaries.append(summary)

# Check that the Summary Files Exist. 
if not all_summaries:
    log(LogLevel.ERROR, "baseline:aggregation", "No Summary Files Found! Run Steps 1 and 2 First.")
else:
    log(LogLevel.INFO, "baseline:aggregation", "Loaded %d Summary File(s) Across All Parts.\n", len(all_summaries))

    # Store the Records Togther.
    records = []
    
    # Aggregate Key Metrics Across all Parts.
    for s in all_summaries:
        records.append({
            "Part"              : s["part"],
            "Min Score"         : s.get("min_det_score", MIN_SCORE),
            "FPPI"              : s.get("fppi"),
            "AFDR"              : s.get("fl_dr_in_range"),
            "HFAR"              : s.get("far"),
            "Detected Aircraft" : s.get("det_aircraft_in_range"),
            "Total Aircraft"    : s.get("tot_aircraft_in_range"),
            "EDR (All)"         : s.get("Tracking", {}).get("Encounters", {}).get("700", {}).get("All", {}).get("dr"),
            "EDR (Above)"       : s.get("Tracking", {}).get("Encounters", {}).get("700", {}).get("Above Horizon", {}).get("dr"),
            "EDR (Below)"       : s.get("Tracking", {}).get("Encounters", {}).get("700", {}).get("Below Horizon", {}).get("dr"),
        })

    # Display the Results for Each Part of the Dataset. 
    df_parts = pd.DataFrame(records)
    log(LogLevel.INFO, "baseline:aggregation", "Per-Part Results:")
    display(df_parts)

### **Baseline Benchmark Summary**

Display the aggregated statistics across all three parts of the dataset in order to provide an overall baseline performance for the full hard dataset. Additionally, statistics for each individual subset of the hard dataset are also displayed for more detailed analysis.

In [ ]:
# Check that the Summary Files Exist. 
if not all_summaries:
    log(LogLevel.ERROR, "baseline:summary", "No Summary Files Found! Run Steps 1, 2 and 3 First.")
else:
    # # Aggregate Metrics Across All Parts.
    total_detected   = df_parts["Detected Aircraft"].sum()
    total_aircraft   = df_parts["Total Aircraft"].sum()
    mean_fppi        = df_parts["FPPI"].mean()
    mean_hfar        = df_parts["HFAR"].mean()
    overall_afdr     = total_detected / total_aircraft if total_aircraft > 0 else 0
    mean_edr_all     = df_parts["EDR (All)"].mean()
    mean_edr_above   = df_parts["EDR (Above)"].mean()
    mean_edr_below   = df_parts["EDR (Below)"].mean()

    # Rank Metrics Against Thresholds and Display Results.
    if mean_fppi <= 0.0005:
        ranked_fppi = log(LogLevel.INFO, "baseline:summary", "FPPI Within Acceptable Limit (≤ 0.0005).")
    else:
        ranked_fppi = log(LogLevel.WARNING, "baseline:summary", f"FPPI Exceeds Limit! Actual: {mean_fppi:.6f} (≤ 0.0005).")
    if mean_hfar <= 0.5:
        ranked_hfar = log(LogLevel.INFO, "baseline:summary", "HFAR Within Acceptable Limit (≤ 0.5).")
    else:
        ranked_hfar = log(LogLevel.WARNING, "baseline:summary", f"HFAR Exceeds Limit! Actual: {mean_hfar:.4f} (≤ 0.5).")

    # Build a Summary DataFrame for Display and CSV Export.
    summary_df = pd.DataFrame([
        {"Metric": "AFDR (Airborne Flight Detection Rate)",  "Value": f"{overall_afdr:.4f}",  "Status": ""},
        {"Metric": "FPPI (False Positives Per Image)",       "Value": f"{mean_fppi:.6f}",     "Status": ranked_fppi},
        {"Metric": "HFAR (Horizontal False Alarm Rate)",     "Value": f"{mean_hfar:.4f}",     "Status": ranked_hfar},
        {"Metric": "EDR - All Scenarios",                    "Value": f"{mean_edr_all:.4f}",  "Status": ""},
        {"Metric": "EDR - Above Horizon",                    "Value": f"{mean_edr_above:.4f}","Status": ""},
        {"Metric": "EDR - Below Horizon",                    "Value": f"{mean_edr_below:.4f}","Status": ""},
        {"Metric": "Total Aircraft Detected",                "Value": f"{int(total_detected)}","Status": ""},
        {"Metric": "Total Aircraft in Dataset",              "Value": f"{int(total_aircraft)}","Status": ""},
    ])

    # Print Summary to Console.
    print("\n" + "="*60)
    print(" BASELINE MODEL - FULL HARD DATASET BENCHMARK SUMMARY")
    print(" Model: Dmytro Poplavskiy Winning Solution (AICrowd 2021)")
    print("="*60)
    display(summary_df.set_index("Metric"))

    # Save Aggregated Summary to CSV.
    output_path = os.path.join(SUMMARIES_DIR, "baseline_benchmark_summary.csv")
    summary_df.to_csv(output_path, index=False)
    print()
    log(LogLevel.INFO, "baseline:summary", "Summary saved to: %s", output_path)

<ERROR><baseline:summary> No Summary Files Found! Run Steps 1, 2 and 3 First.



---

<br/>

## **Hard Dataset Results**

The table below reports baseline performance broken down by scenario category across the custom
hard dataset (1,618 flights). Flights were selected from the full AOT dataset using configurable
filter criteria to isolate the most challenging sequences, providing a more targeted benchmark
than the full competition dataset.

| Scenario                     | Flights | FPPI | AFDR | HFAR | EDR |
|------------------------------|---------|------|------|------|-----|
| **Overall (All Criteria)**   |**1,618**| —    | —    | —    | —   |
| Small Object (bbox ≤ 36 px²) | 662     | —    | —    | —    | —   |
| Below Horizon                | 1,053   | —    | —    | —    | —   |
| Multi-Object                 | 627     | —    | —    | —    | —   |
| No Visible Object            | 354     | —    | —    | —    | —   |

---